# Alignment v3 results

This notebook is observational: Slurm jobs write canonical artifacts and execute only the cell associated with their stage. COCO validation results appear only after the final recipe is locked.

In [1]:
from pathlib import Path
import json, os
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
ROOT = Path.cwd()
RESULTS = ROOT / 'results/alignment_v3'
STAGE = os.environ.get('ALIGNMENT_V3_NOTEBOOK_STAGE', 'all')
ARRAY_INDEX = os.environ.get('ALIGNMENT_V3_NOTEBOOK_INDEX')
def read_json(path):
    return json.loads(path.read_text()) if path.is_file() else None
def read_csv(path):
    return pd.read_csv(path) if path.is_file() else pd.DataFrame()
def pending(message):
    display(Markdown(f'> **Pending:** {message}'))
display(Markdown(f'Notebook stage: `{STAGE}`; array index: `{ARRAY_INDEX}`'))

Notebook stage: `report`; array index: `None`

In [ ]:
validation = read_json(RESULTS / 'manifests/validation.json')
split = read_json(RESULTS / 'manifests/split.json')
prefetch = read_json(RESULTS / 'prefetch.json')
display(Markdown('## Validation, leakage controls and model prefetch'))
display(pd.DataFrame([validation]) if validation else Markdown('Validation pending.'))
display(pd.DataFrame([split]) if split else Markdown('Split pending.'))
if prefetch: display(pd.DataFrame([prefetch]))

In [ ]:
display(Markdown('## Locally measured paired references'))
rows = [read_json(path) for path in sorted((RESULTS / 'references').glob('*/metrics.json'))]
rows = [row for row in rows if row]
if rows:
    reference_frame = pd.DataFrame(rows)
    display(reference_frame[[c for c in ['experiment_id','i2t_R@1','t2i_R@1','mean_R@1','bidirectional_pair_latency_ms','params_total_inference'] if c in reference_frame]])
else: pending('Reference evaluation has not completed.')

In [ ]:
display(Markdown('## RTX PRO 6000 Blackwell batch profiling'))
profiles = read_csv(RESULTS / 'batch_probe/profiles.csv')
common = read_json(RESULTS / 'batch_probe/common_batch.json')
if profiles.empty: pending('Batch profiling has not completed.')
else:
    display(pd.DataFrame([common]))
    display(profiles)
    for name, group in profiles.groupby('experiment_id'):
        group.plot(x='batch_size', y='images_per_second', marker='o', title=name, figsize=(7,3)); plt.show()

In [ ]:
display(Markdown('## Six-pair full-schedule comparison'))
rows = [read_json(path) for path in sorted((RESULTS / 'pair').glob('*/metrics.json'))]
rows = [row for row in rows if row]
if rows:
    pair_frame = pd.DataFrame(rows).sort_values('mean_R@1', ascending=False)
    display(pair_frame[[c for c in ['experiment_id','seed','mean_R@1','bidirectional_pair_latency_ms','params_total_inference','params_trainable_inference'] if c in pair_frame]])
else: pending('Pair training/evaluation has not completed.')
selection = read_json(RESULTS / 'selection/pair.json')
if selection: display(Markdown('### Pre-registered pair gate')); display(pd.DataFrame([selection]))

In [ ]:
display(Markdown('## MobileCLIP2 teacher cache integrity'))
cache = read_json(RESULTS / 'teacher_cache/mobileclip2_s0_dfndr2b.metadata.json')
display(pd.DataFrame([cache]) if cache else Markdown('Teacher cache pending or gated off.'))

In [ ]:
display(Markdown('## Distillation-strength sensitivity'))
rows = [read_json(path) for path in sorted((RESULTS / 'sensitivity').glob('*/metrics.json'))]
rows = [row for row in rows if row]
if rows:
    sensitivity = pd.DataFrame(rows)
    display(sensitivity.groupby('experiment_id')['mean_R@1'].agg(['mean','std','count']).reset_index())
else: pending('Distillation sensitivity is pending or gated off.')
selection = read_json(RESULTS / 'selection/distillation.json')
if selection: display(pd.DataFrame([selection]))

In [ ]:
display(Markdown('## Optional-component smoke tests and two-seed ablations'))
smoke = read_json(RESULTS / 'component_smoke.json')
if smoke: display(pd.DataFrame(smoke.get('components', [])))
rows = [read_json(path) for path in sorted((RESULTS / 'ablation').glob('*/metrics.json'))]
rows = [row for row in rows if row]
if rows:
    ablations = pd.DataFrame(rows)
    display(ablations.groupby('experiment_id')['mean_R@1'].agg(['mean','std','count']).reset_index().sort_values('mean', ascending=False))
else: pending('Ablations are pending or gated off.')
recipe = read_json(RESULTS / 'selection/recipe.json')
if recipe: display(Markdown('### Locked recipe')); display(pd.DataFrame([recipe]))

In [ ]:
display(Markdown('## Locked full-data three-seed result'))
rows = [read_json(path) for path in sorted((RESULTS / 'final').glob('*/metrics.json'))]
rows = [row for row in rows if row]
if rows:
    final = pd.DataFrame(rows)
    display(final)
    display(final[['mean_R@1','i2t_R@1','t2i_R@1']].agg(['mean','std','min','max']))
else: pending('Sealed final evaluation has not run.')
transfer_rows = [read_json(path) for path in sorted((RESULTS / 'transfer').glob('*/metrics.json'))]
transfer_rows = [row for row in transfer_rows if row]
if transfer_rows: display(Markdown('### Optional Flickr30k transfer')); display(pd.DataFrame(transfer_rows))

In [ ]:
display(Markdown('## Oracle complementarity diagnostic'))
value = read_json(RESULTS / 'oracle.json')
display(pd.DataFrame([value]) if value else Markdown('Oracle analysis pending.'))
display(Markdown('*This development-set expert oracle is an upper bound, not a learned or deployable router result.*'))

In [2]:
display(Markdown('## Final measured report and literature context'))
measured = read_csv(RESULTS / 'measured_results.csv')
seed_statistics = read_csv(RESULTS / 'seed_statistics.csv')
literature = read_csv(RESULTS / 'literature_context.csv')
report = read_json(RESULTS / 'report.json')
if report: display(pd.DataFrame([report]))
display(Markdown('### Locally measured'))
display(measured if not measured.empty else Markdown('Measured report pending.'))
if not seed_statistics.empty:
    display(Markdown('### Seed statistics (Student-t intervals are unstable at n=2/3)'))
    display(seed_statistics)
display(Markdown('### Literature-only (not locally measured)'))
display(literature if not literature.empty else Markdown('No literature context available.'))

## Final measured report and literature context

,status,measured_rows,final_rows
0,INCOMPLETE,9,0


### Locally measured

,status,source,kind,experiment_id,i2t_R@1,i2t_R@5,i2t_R@10,mean_rank_i2t,median_rank_i2t,t2i_R@1,...,gpu,software,preprocessing,run_id,seed,params_total_training,params_trainable_training,params_training_only,checkpoint,fingerprint_digest
0,COMPLETE,measured_local,paired_reference,mobileclip2_s0_dfndr2b,0.6312,0.8478,0.9118,5.238200,1.0,0.428440,...,NVIDIA RTX PRO 6000 Blackwell Server Edition,"{'python': '3.9.25', 'platform': 'Linux-5.14.0...","{'model_name': 'MobileCLIP2-S0', 'pretrained':...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,COMPLETE,measured_local,paired_reference,openclip_vit_b32_quickgelu_openai,0.5002,0.7496,0.8324,9.214800,1.0,0.303550,...,NVIDIA RTX PRO 6000 Blackwell Server Edition,"{'python': '3.9.25', 'platform': 'Linux-5.14.0...","{'model_name': 'ViT-B-32-quickgelu', 'pretrain...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,COMPLETE,measured_local,paired_reference,siglip2_vit_b32_256_webli,0.6616,0.8636,0.9208,4.622200,1.0,0.476013,...,NVIDIA RTX PRO 6000 Blackwell Server Edition,"{'python': '3.9.25', 'platform': 'Linux-5.14.0...","{'model_name': 'ViT-B-32-SigLIP2-256', 'pretra...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,COMPLETE,measured_local,pair,dinov3_convnext_tiny__all_minilm_l6_v2,0.1732,0.4040,0.5306,37.008598,9.0,0.148535,...,NVIDIA RTX PRO 6000 Blackwell Server Edition,NaN,NaN,dinov3_convnext_tiny__all_minilm_l6_v2__seed_42,42.0,52454883.0,1921539.0,0.0,checkpoints/alignment_v3/pair/dinov3_convnext_...,0fce6f251fd0dbe72c9c2db5f539c78a9f00cdd145e049...
4,COMPLETE,measured_local,pair,dinov3_convnext_tiny__bge_small_en,0.1524,0.3914,0.5162,39.409401,10.0,0.131502,...,NVIDIA RTX PRO 6000 Blackwell Server Edition,NaN,NaN,dinov3_convnext_tiny__bge_small_en__seed_42,42.0,63101667.0,1921539.0,0.0,checkpoints/alignment_v3/pair/dinov3_convnext_...,9b36560b29e8376fc6317435df1b05b9d991d1f21b7334...
5,COMPLETE,measured_local,pair,dinov3_convnext_tiny__e5_small_v2,0.1580,0.3824,0.5112,41.515800,10.0,0.126744,...,NVIDIA RTX PRO 6000 Blackwell Server Edition,NaN,NaN,dinov3_convnext_tiny__e5_small_v2__seed_42,42.0,63101667.0,1921539.0,0.0,checkpoints/alignment_v3/pair/dinov3_convnext_...,574e43898b4078e4be66e6a1a9b7243b5b1362c7fe39e4...
6,COMPLETE,measured_local,pair,dinov3_vits16__all_minilm_l6_v2,0.1908,0.4454,0.5726,29.865801,7.0,0.149534,...,NVIDIA RTX PRO 6000 Blackwell Server Edition,NaN,NaN,dinov3_vits16__all_minilm_l6_v2__seed_42,42.0,45778563.0,1478403.0,0.0,checkpoints/alignment_v3/pair/dinov3_vits16__a...,476214d8091479cf1a1faaa90336c5149d198cb4eb1550...
7,COMPLETE,measured_local,pair,dinov3_vits16__bge_small_en,0.1958,0.4464,0.5680,32.032600,7.0,0.143417,...,NVIDIA RTX PRO 6000 Blackwell Server Edition,NaN,NaN,dinov3_vits16__bge_small_en__seed_42,42.0,56425347.0,1478403.0,0.0,checkpoints/alignment_v3/pair/dinov3_vits16__b...,34f608deed82c190ba44514ee9d7b54e8c6a88a75d39d6...
8,COMPLETE,measured_local,pair,dinov3_vits16__e5_small_v2,0.1860,0.4282,0.5594,32.607201,8.0,0.137579,...,NVIDIA RTX PRO 6000 Blackwell Server Edition,NaN,NaN,dinov3_vits16__e5_small_v2__seed_42,42.0,56425347.0,1478403.0,0.0,checkpoints/alignment_v3/pair/dinov3_vits16__e...,3ad05a9d2a647a3da11806aa6f21a3dc0eb8129dfa091c...


### Seed statistics (Student-t intervals are unstable at n=2/3)

,kind,experiment_id,mean,std,count,min,max,ci95_halfwidth,ci_note
0,pair,dinov3_convnext_tiny__all_minilm_l6_v2,0.160867,NaN,1,0.160867,0.160867,NaN,not reported
1,pair,dinov3_convnext_tiny__bge_small_en,0.141951,NaN,1,0.141951,0.141951,NaN,not reported
2,pair,dinov3_convnext_tiny__e5_small_v2,0.142372,NaN,1,0.142372,0.142372,NaN,not reported
3,pair,dinov3_vits16__all_minilm_l6_v2,0.170167,NaN,1,0.170167,0.170167,NaN,not reported
4,pair,dinov3_vits16__bge_small_en,0.169608,NaN,1,0.169608,0.169608,NaN,not reported
5,pair,dinov3_vits16__e5_small_v2,0.161790,NaN,1,0.161790,0.161790,NaN,not reported
6,paired_reference,mobileclip2_s0_dfndr2b,0.529820,NaN,1,0.529820,0.529820,NaN,not reported
7,paired_reference,openclip_vit_b32_quickgelu_openai,0.401875,NaN,1,0.401875,0.401875,NaN,not reported
8,paired_reference,siglip2_vit_b32_256_webli,0.568807,NaN,1,0.568807,0.568807,NaN,not reported


### Literature-only (not locally measured)

,id,source,note
0,dinov2_large_all_roberta_large,literature,cited ceiling; never represented as locally ev...
